## Problem Statement

## Dataset Overview

In [37]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix


## loading the dataset

In [38]:
# the dataset is loaded
df = pd.read_csv('/Users/jameswoodthorpe/Code/University/Year_3/6033/Data/spam.csv', encoding='latin-1')

In [39]:
print(df.head())

     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  


#### dataset evaluations:
<ul>
    <li> 1
    <li> 2
    <li> 3
    <li>
<ul>



## cleaning the data

In [40]:
# this function cleans the text. firstly it is set to lower case to it can be easily parsed. then it is punctuation and special characters are removed
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z]", " ", text)
    return text

In [41]:
df['v1'] = df['v1'].map({'ham': 0, 'spam': 1})


In [42]:
df['v2'] = df['v2'].apply(clean_text)

x_train, x_test, y_train, y_test = train_test_split(
    df['v2'],
    df['v1'],
    test_size=0.2,
    random_state=42,
    stratify=df['v1']
)

## vectorization

In [43]:
vectorizer = TfidfVectorizer(stop_words='english')

x_train_tfidf = vectorizer.fit_transform(x_train)

x_test_tfidf = vectorizer.transform(x_test)

print(f"Vocabulary size: {len(vectorizer.get_feature_names_out())}")

Vocabulary size: 6575


## initializing and training

In [44]:
# initialize and train
model = MultinomialNB()
model.fit(x_train_tfidf, y_train)

# make predictions
y_pred = model.predict(x_test_tfidf)

# evaluate
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[965   1]
 [ 36 113]]
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       966
           1       0.99      0.76      0.86       149

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.96      1115



In [45]:
def predict_spam(new_message):
    # cleans the input
    cleaned_msg = clean_text(new_message)

    # vectorize the input
    vectorized_msg = vectorizer.transform([cleaned_msg])

    prediction = model.predict(vectorized_msg)

    return "SPAM" if prediction[0] == 1 else "HAM"

# test inputs to verify the model succeeds
print(predict_spam("Congratulations! You've won a $500 Amazon gift card. Click here to claim."))
print(predict_spam("Hey, are we still meeting for coffee at 3pm?"))
print(predict_spam("Hey, I'm running about 10 minutes late for our meeting, see you soon!"))


SPAM
HAM
HAM


In [46]:
import joblib

# Save the model and the vectorizer to your project folder
joblib.dump(model, 'spam_model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')

['vectorizer.pkl']